# SRFD-DETR workflow (Colab / Jupyter outline)

This notebook is a **reproducibility-oriented outline** for reviewers: environment setup → optional Hugging Face weights → MEG preprocessing → detector training.

**GPU:** MEG `batch_infer.py` expects CUDA. Colab: Runtime → Change runtime type → GPU.

## 1) Install dependencies

- Core detector: `requirements.txt`
- MEG: `requirements-meg.txt` (diffusers, transformers, …)

In [ ]:
# Colab: mount Drive if your code/weights live there
# from google.colab import drive
# drive.mount('/content/drive')

import sys, subprocess

def pip_install(req_file):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", req_file])

# Adjust REPO_ROOT to where you cloned this repository
REPO_ROOT = "/content/SRFD-DETR"  # example

pip_install(REPO_ROOT + "/requirements.txt")
pip_install(REPO_ROOT + "/requirements-meg.txt")

## 2) Get code

- **Option A:** `git clone <your-public-repo-url>` then set `REPO_ROOT`.
- **Option B:** Upload a zip and unzip under `/content`.

Verify:
```
ls $REPO_ROOT/MEG
ls $REPO_ROOT/ultralytics/cfg/models/rt-detr
```

## 3) Hugging Face checkpoints (MEG)

Recommended public weights (same family as paper description):

- [runwayml/stable-diffusion-v1-5](https://huggingface.co/runwayml/stable-diffusion-v1-5)
- [lllyasviel/sd-controlnet-canny](https://huggingface.co/lllyasviel/sd-controlnet-canny)

**Offline / cache:** either login for gated models or pre-upload snapshot folders to Drive and pass `--sd_dir` / `--controlnet_dir`.

Your scripts often use `local_files_only=True`; then paths must be **local snapshot directories**, not only repo ids.

In [ ]:
# Optional: Hugging Face token for gated downloads (Colab secrets recommended)
# import os
# os.environ["HF_TOKEN"] = "..."  # or huggingface-cli login

from huggingface_hub import snapshot_download

CACHE = REPO_ROOT + "/hf_cache"  # example

# Uncomment to download snapshots (large disk usage)
# sd_path = snapshot_download("runwayml/stable-diffusion-v1-5", cache_dir=CACHE)
# cn_path = snapshot_download("lllyasviel/sd-controlnet-canny", cache_dir=CACHE)
# print("SD:", sd_path)
# print("CN:", cn_path)

## 4) MEG batch inference (preprocess SEM folder)

Runs `MEG/batch_infer.py`: structural control map → DDIM img2img → K candidates → edge IoU + tenengrad fusion.

Replace paths with your local snapshot directories.

In [ ]:
import subprocess, shlex

SD_DIR = REPO_ROOT + "/hf_cache/models--runwayml--stable-diffusion-v1-5/snapshots/..."  # fill
CN_DIR = REPO_ROOT + "/hf_cache/models--lllyasviel--sd-controlnet-canny/snapshots/..."  # fill
INPUT_DIR = REPO_ROOT + "/dataset/examples_raw"   # put a few sample images
OUT_DIR = REPO_ROOT + "/dataset/examples_meg"

cmd = f"""python "{REPO_ROOT}/MEG/batch_infer.py" \
  --sd_dir "{SD_DIR}" \
  --controlnet_dir "{CN_DIR}" \
  --input_dir "{INPUT_DIR}" \
  --out_dir "{OUT_DIR}" \
  --samples 2 --steps 12"""
print(cmd)
# subprocess.check_call(cmd, shell=True)  # uncomment when paths are valid

## 5) Train SRFD-DETR detector

Prepare `data.yaml` (YOLO format). Domain labels are inferred from image paths in `ultralytics/data/dataset.py` — align folder names with your convention.

```bash
python train.py --data /path/to/data.yaml --cfg ultralytics/cfg/models/rt-detr/rtdetr-r18.yaml
```

In [ ]:
train_cmd = f'python "{REPO_ROOT}/train.py" --data "{REPO_ROOT}/dataset/data.yaml" --epochs 1 --batch 2 --name colab_smoke'
print(train_cmd)
# subprocess.check_call(train_cmd, shell=True)  # uncomment when data.yaml exists

## 6) Validation

```bash
python val.py --weights runs/train/.../weights/best.pt --data path/to/data.yaml --split test
```

Or in Python:

```python
from ultralytics import RTDETR
m = RTDETR("runs/train/.../weights/best.pt")
m.val(data="path/to/data.yaml", split="test")
```